# Getting started with pounce

`pounce` is a Rust port of Ipopt for nonlinear optimization. The Python interface gives you two entry points:

* **`pounce.Problem`** — the cyipopt-shaped object-oriented API. You hand the solver a user object with `objective`/`gradient`/`constraints`/`jacobian`/`hessian` methods.
* **`pounce.minimize`** — the scipy-style functional facade. Pass `fun`, `jac`, `hess`, `bounds`, `constraints` directly.

Both build on the same Rust solver. This notebook walks both paths plus the warm-start convention.

We'll use **HS071** — the canonical Ipopt test problem — throughout:

$$\min_{x \in \mathbb{R}^4} \; x_1 x_4 (x_1 + x_2 + x_3) + x_3$$

$$\text{s.t.} \quad x_1 x_2 x_3 x_4 \geq 25, \quad x_1^2 + x_2^2 + x_3^2 + x_4^2 = 40, \quad 1 \leq x_i \leq 5.$$

Optimum: $x^\star \approx (1.000, 4.743, 3.821, 1.379)$, $f^\star \approx 17.014$.

In [1]:
import numpy as np

import pounce
print(f"pounce {pounce.__version__}")

pounce 0.1.0


## 1. The `Problem` API (cyipopt-style)

Build a class whose methods match the cyipopt calling convention. The solver detects whether you supplied an exact Hessian via `hasattr(obj, 'hessian')` and falls back to L-BFGS otherwise.

Sparse Jacobians and Hessians use the `(rows, cols)` structure / `values` value split, also cyipopt-style:

In [2]:
class HS071:
    def objective(self, x):
        return x[0] * x[3] * (x[0] + x[1] + x[2]) + x[2]

    def gradient(self, x):
        return np.array([
            x[0] * x[3] + x[3] * (x[0] + x[1] + x[2]),
            x[0] * x[3],
            x[0] * x[3] + 1.0,
            x[0] * (x[0] + x[1] + x[2]),
        ])

    def constraints(self, x):
        return np.array([np.prod(x), np.dot(x, x)])

    def jacobianstructure(self):
        # Dense 2-by-4: both constraints depend on all variables.
        return (np.repeat([0, 1], 4), np.tile([0, 1, 2, 3], 2))

    def jacobian(self, x):
        return np.array([
            x[1] * x[2] * x[3], x[0] * x[2] * x[3],
            x[0] * x[1] * x[3], x[0] * x[1] * x[2],
            2 * x[0], 2 * x[1], 2 * x[2], 2 * x[3],
        ])

In [3]:
prob = pounce.Problem(
    n=4, m=2, problem_obj=HS071(),
    lb=[1.0] * 4, ub=[5.0] * 4,
    cl=[25.0, 40.0], cu=[2e19, 40.0],
)
prob.add_option("tol", 1e-8)
prob.add_option("print_level", 0)

x, info = prob.solve(x0=np.array([1.0, 5.0, 5.0, 1.0]))
print(f"status: {info['status_msg']}")
print(f"x*    : {x}")
print(f"f*    : {info['obj_val']:.6f}")
print(f"iters : {info['iter_count']}")

status: Solve_Succeeded
x*    : [0.99999999 4.74299964 3.82114998 1.37940829]
f*    : 17.014017
iters : 11


### What's in `info`?

Everything you'd expect from a KKT solve:

* `status`, `status_msg` — exit code / human-readable reason
* `obj_val`, `g` — final objective and constraint residuals
* `mult_g` — equality / inequality constraint multipliers (the Lagrange $\lambda$)
* `mult_x_L`, `mult_x_U` — bound multipliers ($z^L$, $z^U$)
* `iter_count` — iteration count

In [4]:
for k, v in info.items():
    print(f"{k:>12} : {v}")

      status : 0
  status_msg : Solve_Succeeded
     obj_val : 17.01401714521598
           g : [24.99999975 40.        ]
      mult_g : [-0.55229366  0.16146856]
    mult_x_L : [1.08787123e+00 6.69490727e-10 8.88256054e-10 6.60476737e-09]
    mult_x_U : [6.26475881e-10 9.75058202e-09 2.12571864e-09 6.92125411e-10]
  iter_count : 11


## 2. The `minimize` facade (scipy-style)

Closer to `scipy.optimize.minimize`. Constraints are dicts with `'type': 'eq' | 'ineq'`. Returns an `OptimizeResult`-shaped dataclass.

In [5]:
def f(x):
    return float(x @ x)

def grad(x):
    return 2.0 * x

def c_fun(x):
    return np.array([x[0] + x[1] - 1.0])

def c_jac(x):
    return np.array([[1.0, 1.0]])

res = pounce.minimize(
    f, x0=np.zeros(2), jac=grad,
    constraints=[{"type": "eq", "fun": c_fun, "jac": c_jac}],
    options={"tol": 1e-10, "print_level": 0},
)
print(f"success: {res.success}")
print(f"x*     : {res.x}")
print(f"f*     : {res.fun:.6f}")
print(f"iters  : {res.nit}")

success: True
x*     : [0.5 0.5]
f*     : 0.500000
iters  : 2


## 3. L-BFGS path (no Hessian supplied)

Omit `hessian`/`hessianstructure` from your problem object and the solver runs with a limited-memory quasi-Newton approximation. The same `Problem` interface — pounce auto-switches based on what your object provides.

In [6]:
class HS071_no_hess(HS071):
    pass  # No hessian / hessianstructure → L-BFGS path.

prob = pounce.Problem(
    n=4, m=2, problem_obj=HS071_no_hess(),
    lb=[1.0] * 4, ub=[5.0] * 4,
    cl=[25.0, 40.0], cu=[2e19, 40.0],
)
print(f"has_hessian: {prob.has_hessian}")

prob.add_option("tol", 1e-8)
prob.add_option("print_level", 0)
x_lbfgs, info_lbfgs = prob.solve(x0=np.array([1.0, 5.0, 5.0, 1.0]))
print(f"status: {info_lbfgs['status_msg']}, f* = {info_lbfgs['obj_val']:.6f}, iters = {info_lbfgs['iter_count']}")

has_hessian: False
status: Solve_Succeeded, f* = 17.014017, iters = 11


## 4. Warm-starting

Re-solving from a primal point close to the optimum cuts iterations even with the default initializer. To also forward the dual seeds (`lagrange`, `zl`, `zu`) into the iterate initializer, set `warm_start_init_point=yes`:

In [7]:
def make(extra=None):
    p = pounce.Problem(
        n=4, m=2, problem_obj=HS071(),
        lb=[1.0] * 4, ub=[5.0] * 4,
        cl=[25.0, 40.0], cu=[2e19, 40.0],
    )
    p.add_option("tol", 1e-8)
    p.add_option("print_level", 0)
    for k, v in (extra or {}).items():
        p.add_option(k, v)
    return p

cold_x, cold_info = make().solve(x0=np.array([1.0, 5.0, 5.0, 1.0]))

warm_x, warm_info = make({"warm_start_init_point": "yes"}).solve(
    x0=cold_x,
    lagrange=np.asarray(cold_info["mult_g"]),
    zl=np.asarray(cold_info["mult_x_L"]),
    zu=np.asarray(cold_info["mult_x_U"]),
)

print(f"cold: {cold_info['iter_count']} iters")
print(f"warm: {warm_info['iter_count']} iters")
print(f"|\u0394x|  = {np.max(np.abs(warm_x - cold_x)):.2e}")

cold: 11 iters
warm: 11 iters
|Δx|  = 7.94e-13


## 5. Setting solver options

Any Ipopt option name works. Strings, ints, floats, and bools are routed to the right setter automatically.

Common ones:

| Option | Effect |
|---|---|
| `tol` | Convergence tolerance on the KKT error |
| `max_iter` | Iteration cap |
| `print_level` | 0 = silent, 5 = default, 12 = noisy |
| `hessian_approximation` | `"exact"` (default) or `"limited-memory"` |
| `linear_solver` | Backend (`"feral"` is the pure-Rust default) |
| `mu_strategy` | `"monotone"` or `"adaptive"` |

See the [Ipopt options reference](https://coin-or.github.io/Ipopt/OPTIONS.html) for the full list.

In [8]:
prob = pounce.Problem(
    n=4, m=2, problem_obj=HS071(),
    lb=[1.0] * 4, ub=[5.0] * 4,
    cl=[25.0, 40.0], cu=[2e19, 40.0],
)
prob.add_option("tol", 1e-12)
prob.add_option("max_iter", 200)
prob.add_option("mu_strategy", "adaptive")
prob.add_option("print_level", 0)

x, info = prob.solve(x0=np.array([1.0, 5.0, 5.0, 1.0]))
print(info["status_msg"], info["iter_count"], info["obj_val"])

Search_Direction_Becomes_Too_Small 12 17.014017140224173
